# Stock Trend Prediction - Exploratory Analysis
This notebook demonstrates the complete workflow for stock trend prediction using LSTM.

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import load_stock_data
from src.features import prepare_features
from src.model import build_lstm_model, prepare_data

sns.set_style('whitegrid')
%matplotlib inline

## 1. Load Data

In [ ]:
ticker = 'RELIANCE.NS'
df = load_stock_data(ticker, period='5y')
print(f"Loaded {len(df)} days of data")
df.head()

## 2. Visualize Stock Price

In [ ]:
plt.figure(figsize=(14, 6))
plt.plot(df['Date'], df['Close'])
plt.title(f'{ticker} - Closing Price')
plt.xlabel('Date')
plt.ylabel('Price')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 3. Add Technical Indicators

In [ ]:
df, feature_cols = prepare_features(df)
print(f"Features: {feature_cols}")
print(f"Data shape after feature engineering: {df.shape}")
df.head()

## 4. Visualize Technical Indicators

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 12))

# SMA
axes[0].plot(df['Date'], df['Close'], label='Close', alpha=0.7)
axes[0].plot(df['Date'], df['SMA20'], label='SMA20', alpha=0.7)
axes[0].plot(df['Date'], df['SMA50'], label='SMA50', alpha=0.7)
axes[0].set_title('Simple Moving Averages')
axes[0].legend()
axes[0].grid(True)

# RSI
axes[1].plot(df['Date'], df['RSI'], label='RSI', color='purple')
axes[1].axhline(y=70, color='r', linestyle='--', label='Overbought')
axes[1].axhline(y=30, color='g', linestyle='--', label='Oversold')
axes[1].set_title('Relative Strength Index (RSI)')
axes[1].legend()
axes[1].grid(True)

# MACD
axes[2].plot(df['Date'], df['MACD'], label='MACD', color='blue')
axes[2].plot(df['Date'], df['MACD_Signal'], label='Signal', color='red')
axes[2].set_title('MACD')
axes[2].legend()
axes[2].grid(True)

plt.tight_layout()
plt.show()

## 5. Target Distribution

In [ ]:
target_counts = df['Target'].value_counts()
print(f"Target distribution:\n{target_counts}")
print(f"\nPercentage:\n{target_counts / len(df) * 100}")

plt.figure(figsize=(8, 5))
target_counts.plot(kind='bar')
plt.title('Target Distribution (0=Down, 1=Up)')
plt.xlabel('Target')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.show()

## 6. Correlation Matrix

In [ ]:
plt.figure(figsize=(12, 8))
correlation = df[feature_cols + ['Target']].corr()
sns.heatmap(correlation, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

## 7. Prepare Data for LSTM

In [ ]:
X_train, X_test, y_train, y_test, scaler = prepare_data(df, feature_cols)

print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")
print(f"Input shape: {X_train.shape}")
print(f"\nTrain target distribution: {np.bincount(y_train)}")
print(f"Test target distribution: {np.bincount(y_test)}")

## 8. Build and Train Model

In [ ]:
model = build_lstm_model((X_train.shape[1], X_train.shape[2]))
model.summary()

In [ ]:
history = model.fit(
    X_train, y_train,
    epochs=10,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

## 9. Evaluate Model

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred = (model.predict(X_test) > 0.5).astype(int)

print("Classification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
cm = confusion_matrix(y_test, y_pred)
print(cm)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

## 10. Training History

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history.history['accuracy'], label='Train')
ax1.plot(history.history['val_accuracy'], label='Validation')
ax1.set_title('Model Accuracy')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(True)

ax2.plot(history.history['loss'], label='Train')
ax2.plot(history.history['val_loss'], label='Validation')
ax2.set_title('Model Loss')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

## 11. Save Model

In [ ]:
model.save('../stock_lstm_model.h5')
print("Model saved successfully!")